# **KFO5001 - Exploratory Data Analysis; From Patterns to Insights**

> **Public data notice:** All distributed recordings are independently generated synthetic surrogates, not experimental observations. They preserve the workshop structure only and must not be used for scientific inference. See `DATA_NOTICE.md`.


1. For a personalized copy, remember to **Save a copy in Drive**.
2. Complete the five-minute warm-up below.
3. Run the setup cell and setup check.
4. Follow the numbered sections in order.

After a runtime reset, rerun setup and setup check. Each section rebuilds its own isolated analysis state.

## Warm-up: Colab, Python, and pandas in five minutes

* Run a cell with **Shift+Enter**.
* Cells remember values and tables created above them.

This dummy freezing example is independent of the workshop data. Try change one number, rerun the cells below, and observe what updates.

In [ ]:
subject_id = "subject_A"
pre_tone_freezing = 10
tone_freezing = 60
freezing_change = tone_freezing - pre_tone_freezing

print(subject_id)
print("Freezing change:", freezing_change, "percentage points")

In [ ]:
import pandas as pd

practice_data = pd.DataFrame({
    "subject_id": ["subject_A", "subject_B", "subject_C", "subject_D"],
    "pre_tone_freezing": [10, 20, 15, 30],
    "tone_freezing": [60, 45, 70, 50],
})
practice_data

In [ ]:
practice_data["freezing_change"] = practice_data["tone_freezing"] - practice_data["pre_tone_freezing"]
large_changes = practice_data.loc[practice_data["freezing_change"] >= 40]

display(large_changes)
practice_data.plot.bar(x="subject_id", y="freezing_change", color="#287D74", legend=False, title="Fictional freezing change")

## **Section 0: START/RESTART**

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import importlib
import importlib.util
import os
import random
import subprocess
import sys
import time

REPO_URL = "https://github.com/Hamidreza-Alimohammadi/kfo5001-EDA-workshop.git"
REQUIRED_REPO_FILES = ("KFO5001_EDA_Workshop.ipynb", "data/reduced/sessions.csv")


def running_in_colab():
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


def repo_is_ready(repo_dir):
    return (repo_dir / "workshop_tools").is_dir() and all((repo_dir / path).is_file() for path in REQUIRED_REPO_FILES)


def find_local_repo():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if repo_is_ready(candidate):
            return candidate
    raise RuntimeError("Start JupyterLab from the workshop repository directory.")


IN_COLAB = running_in_colab()
REPO_DIR = Path("/content/kfo5001-EDA-workshop") if IN_COLAB else find_local_repo()


def run_git(command, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            subprocess.run(command, check=True)
            return
        except subprocess.CalledProcessError:
            if attempt == attempts:
                raise
            wait_s = 2 ** attempt + random.uniform(0, 2)
            print(f"Git request failed; retrying in {wait_s:.1f} seconds...")
            time.sleep(wait_s)


def start_session():
    if IN_COLAB:
        if not REPO_DIR.exists():
            wait_s = random.uniform(0, 10)
            print(f"Staggering initial download by {wait_s:.1f} seconds...")
            time.sleep(wait_s)
            run_git(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
        elif repo_is_ready(REPO_DIR):
            try:
                run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
            except (subprocess.CalledProcessError, FileNotFoundError):
                print("Update unavailable; continuing with the valid existing checkout.")
        else:
            raise RuntimeError("The Colab checkout is incomplete. Restart the runtime to obtain a clean copy.")

    if not repo_is_ready(REPO_DIR):
        raise RuntimeError(f"Workshop files are incomplete in {REPO_DIR}.")

    os.chdir(REPO_DIR)

    repo_paths = [path for path in sys.path if "kfo5001-EDA-workshop" in path]
    for repo_path in repo_paths:
        sys.path.remove(repo_path)
    sys.path.insert(0, str(REPO_DIR))

    for name in list(sys.modules):
        if name == "workshop_tools" or name.startswith("workshop_tools."):
            del sys.modules[name]
    importlib.invalidate_caches()

    try:
        revision = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        revision = "unknown"
    location = "Colab checkout" if IN_COLAB else "local checkout"
    print(f"Session ready from {location}: {REPO_DIR} (commit {revision})")


start_session()


def ensure_workshop_packages():
    requirements = {"umap": "umap-learn>=0.5,<1"}
    missing = [requirement for module, requirement in requirements.items() if importlib.util.find_spec(module) is None]
    if not missing:
        return
    if not IN_COLAB:
        raise RuntimeError("Update the local environment with: conda env update -f environment.yml --prune")
    print(f"Installing workshop dependency: {', '.join(missing)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)


ensure_workshop_packages()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from workshop_tools import (
    diagnostics, exploration, features, patterns, psth, robustness,
    session_data, style, synchronization, trajectories,
)

sessions = session_data.load_sessions(has_tracking=True, has_events=True)
prepared_epoch_features = features.load_prepared_features()


### Check the shared preparation

Run this after setup or a runtime reset, also to get an overview of valid subjects, phases, and modalities.

`session_key` chooses one recording for signal-level inspection. Shared tables remain unchanged; results stay inside their `section_N` namespace.


In [ ]:
diagnostics.check_setup(sessions, prepared_epoch_features)

print("Subjects and available phases")
display(session_data.subject_overview(sessions))

print("Modality availability by phase (available sessions / total sessions)")
display(session_data.modality_overview(sessions))


## **Section 1: One experiment, three time scales**

**Goal:** whether tracking, heartbeat-derived HR, and events can be compared directly.


### Select one complete session


In [ ]:
section_1 = SimpleNamespace()
# Session key: (subject_id, phase). Subjects and phases are listed in the setup overview.
section_1.session_key = ("subject_01", "ext_1")
section_1.session = session_data.choose_session(sessions, 
                                                section_1.session_key, 
                                                modalities=("tracking", "freezing", "hr", "events"))
section_1.data = session_data.load_session(section_1.session)

section_1.data.info[[
    "subject_id",
    "phase",
    "duration_s",
    "frame_n",
    "heartbeat_n",
    "event_range_n",
]]


### Compare sampling of modalities

Tracking uses video frames, HR uses heartbeat timestamps, and events are sparse intervals.


In [ ]:
print("Tracking:")
display(section_1.data.tracking.head())

print("Heart rate:")
display(section_1.data.heart_rate.head())

print("Events:")
display(section_1.data.events.head())


In [ ]:
synchronization.sampling_summary(section_1.data.info, section_1.data.tracking, section_1.data.heart_rate, section_1.data.events)


### Inspect variables and missingness


In [ ]:
diagnostics.variable_summary(
    {
        "Tracking": section_1.data.tracking,
        "HeartRate": section_1.data.heart_rate,
        "Events": section_1.data.events,
    },
    transpose=True,
)


To get an understanding of a couple of missing examples:

In [ ]:
diagnostics.missing_examples({
    "Tracking": section_1.data.tracking,
    "HeartRate": section_1.data.heart_rate,
    "Events": section_1.data.events,
})


### Inspect native timestamps

A common pitfall in (blind) analysis: shared `time_s` column does not imply row alignment.


In [ ]:
synchronization.plot_timestamp_geometry(
    section_1.data.tracking,
    section_1.data.heart_rate,
    section_1.data.events,
    trial=1,
)
plt.show()


In [ ]:
# Options: any positive integer; common choices are 20, 40, or 80.
section_1.interval_bins = 50

synchronization.plot_sampling_intervals(
    section_1.data.tracking,
    section_1.data.heart_rate,
    bins=section_1.interval_bins,
    yscale="log",
)
plt.show()


### Align signals to tone onset

The bin size remains explicit because it determines the temporal resolution of the question.


In [ ]:
# Options: any available tone trial, normally 1 to 16.
section_1.selected_trial = 16

# Options: two seconds relative to tone onset, with start less than end.
section_1.aligned_window = (-5, 15)

# Options: 0.25, 0.5, 1, or 2 seconds.
section_1.bin_size_s = 0.25

section_1.binned = synchronization.event_aligned_bins(
    section_1.data.tracking,
    section_1.data.heart_rate,
    section_1.data.events,
    event="tone",
    window=section_1.aligned_window,
    bin_size_s=section_1.bin_size_s,
)

section_1.trial_data = section_1.binned.loc[
    section_1.binned["trial"] == section_1.selected_trial
].copy()

if section_1.trial_data.empty:
    raise ValueError(
        f"Tone trial {section_1.selected_trial} is unavailable."
    )

section_1.trial_data.head()

In [ ]:
section_1.trial_data["bin_center_s"] = (
    section_1.trial_data["bin_start_s"] + section_1.trial_data["bin_end_s"]
) / 2

section_1.features = [
    ("freezing_fraction", "Binned freezing fraction", style.SIGNAL_COLORS["freezing"]),
    ("mean_motion", "Binned motion (a.u.)", style.SIGNAL_COLORS["motion"]),
    ("mean_hr_bpm", "Binned heart rate (bpm)", style.SIGNAL_COLORS["hr"]),
]

section_1.figure, section_1.axes = plt.subplots(1, 3, figsize=(11, 3), sharex=True)

for section_1_axis, (section_1_feature, section_1_title, section_1_color) in zip(section_1.axes, section_1.features):
    section_1_axis.plot(
        section_1.trial_data["bin_center_s"],
        section_1.trial_data[section_1_feature],
        color=section_1_color,
        marker="o",
        markersize=3,
        linewidth=1,
    )
    section_1_axis.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
    section_1_axis.set_xlim(section_1.aligned_window)
    section_1_axis.set_title(section_1_title)
    section_1_axis.set_xlabel("Seconds from tone onset")

section_1.axes[0].set_ylabel("Fraction")
section_1.figure.suptitle(
    f"Tone trial {section_1.selected_trial}: "
    f"{section_1.bin_size_s:g}-second bins"
)
section_1.figure.tight_layout()
plt.show()

**Try:** Change the aligned window or bin size. What detail is gained or lost?


**Section insight:** Align modalities to the same experimental events; use common time bins only for direct bin-wise comparisons, and otherwise preserve native sampling.


## **Section 2: First look across phases**

**Goal:** Inspect complete recordings before averaging across subjects or trials.


### Whole-session traces

Use one `session_key` before expanding to a phase or the whole cohort.


In [ ]:
section_2 = SimpleNamespace()

# Session key: one (subject_id, phase) pair. Use exactly one of "cond", "ext_1", or "ext_2".
section_2.session_key = ("subject_01", "ext_1")
section_2.session = session_data.choose_session(sessions, section_2.session_key)

# Options: None for the complete range, or a positive number such as 1, 1.5, or 2.
section_2.trace_y_limit_n_std = None

section_2.trace_figures = exploration.plot_session_traces(
    sessions,
    selection=section_2.session,
    max_sessions=None,
    y_limit_n_std=section_2.trace_y_limit_n_std,
)
plt.show()


### Event-aligned phase PSTH

Compare motion, freezing, and HR on a common tone-aligned axis.


In [ ]:
# Options: ("all", "cond"), ("all", "ext_1"), or ("all", "ext_2").
section_2.session_selection = ("all", "ext_1")

# Options: "tone"; "shock" is also valid for phases containing shock events.
section_2.alignment_event = "tone"
# Options: any (start, end) tuple in seconds around the alignment event.
section_2.window = (-5, 15)
# Options: 0.05, 0.1, 0.5, or 1 second; 0.1 suits the 30 Hz tracking data.
section_2.bin_size_s = 0.25

# Options: "mean" or "median".
section_2.center = "median"
# Options: "std", "sem", "mad", or None.
section_2.band = "sem"
# Options: True or False.
section_2.show_subject_traces = True
# Options: None for modality colors, or any Matplotlib color such as "0.35".
section_2.subject_trace_color = None
# Options: any number from 0 (transparent) to 1 (opaque).
section_2.subject_trace_alpha = 0.28
# Options: None for the complete range, or a positive number such as 1, 1.5, or 2.
section_2.psth_y_limit_n_std = None

section_2.phase_psth = psth.build_phase_psth(
    sessions,
    selection=section_2.session_selection,
    alignment_event=section_2.alignment_event,
    window=section_2.window,
    bin_size_s=section_2.bin_size_s,
)
section_2.event_windows = psth.build_event_windows(
    sessions,
    selection=section_2.session_selection,
    alignment_event=section_2.alignment_event,
)

psth.plot_phase_psth(
    section_2.phase_psth,
    section_2.event_windows,
    center=section_2.center,
    band=section_2.band,
    show_subject_traces=section_2.show_subject_traces,
    subject_trace_color=section_2.subject_trace_color,
    subject_trace_alpha=section_2.subject_trace_alpha,
    y_limit_n_std=section_2.psth_y_limit_n_std,
    alignment_label=f"{section_2.alignment_event} onset",
)
plt.show()


## **Section 3: From temporal patterns to trial features**

**Goal:** Represent each extinction trial with four interpretable epochs:

```text
Pre-Tone -> Early-Tone -> Late-Tone -> Post-Tone
```


### Inspect one trial before summarizing

Choose a session and trial, then predict which modality changes most across epochs.


In [ ]:
section_3 = SimpleNamespace()
# Session key: one (subject_id, phase) pair from either extinction phase.
section_3.session_key = ("subject_01", "ext_1")
# Options: integers 1 through 16.
section_3.trial = 10
# Options: any positive duration in seconds; 5, 10, or 15 are useful comparisons.
section_3.before_tone_s = 10
# Options: any positive duration in seconds; 5, 10, or 15 are useful comparisons.
section_3.after_tone_s = 10

section_3.session = session_data.choose_session(sessions, section_3.session_key)
features.plot_trial_epochs(
    section_3.session,
    trial=section_3.trial,
    before_s=section_3.before_tone_s,
    after_s=section_3.after_tone_s,
)
plt.show()


### Extract robust features

We need to extract relevant features now, and to start with let's go with the following selection: 


In [ ]:
section_3.extracted_epoch_features = features.extract_epoch_features(
    sessions,
    before_s=section_3.before_tone_s,  # is set from the previous cell
    after_s=section_3.after_tone_s,  # is set from the previous cell
    min_valid_fraction=0.80,
    min_hr_samples=10,
)

section_3.core_feature_columns = [
    "freezing_fraction",
    "longest_freezing_bout_s",
    "latency_to_freezing_s",
    "tone_freezing_observed",
    "motion_median",
    "motion_iqr",
    "motion_p90",
    "speed_median_px_s",
    "distance_px",
    "speed_p90_px_s",
    "hr_median_bpm",
    "hr_p10_bpm",
    "hr_p90_bpm",
    "hr_p90_minus_p10_bpm",
]

section_3.selected_trial_features = section_3.extracted_epoch_features.loc[
    (section_3.extracted_epoch_features["subject_id"] == section_3.session["subject_id"])
    & (section_3.extracted_epoch_features["phase"] == section_3.session["phase"])
    & (section_3.extracted_epoch_features["trial"] == section_3.trial)
]
if section_3.selected_trial_features.empty:
    raise ValueError("No feature rows match the selected subject, phase, and trial.")

section_3.selected_trial_features.loc[:, ["epoch"] + section_3.core_feature_columns].round(2)


One tidy row represents one subject, phase, trial, and epoch.


In [ ]:
section_3.extracted_epoch_features.shape
section_3.extracted_epoch_features.head()

### Compare epochs

- Early-Tone minus Pre-Tone: immediate response
- Late-Tone minus Early-Tone: within-tone adaptation
- Post-Tone minus Late-Tone: immediate recovery
- Post-Tone minus Pre-Tone: persistent change


In [ ]:
section_3.epoch_contrasts = features.build_epoch_contrasts(
    section_3.extracted_epoch_features
)
section_3.selected_trial_contrasts = section_3.epoch_contrasts.loc[
    (section_3.epoch_contrasts["subject_id"] == section_3.session["subject_id"])
    & (section_3.epoch_contrasts["phase"] == section_3.session["phase"])
    & (section_3.epoch_contrasts["trial"] == section_3.trial)
]
if section_3.selected_trial_contrasts.empty:
    raise ValueError("No contrasts match the selected subject, phase, and trial.")

section_3.selected_trial_contrasts.pivot(
    index="feature", columns="contrast", values="value"
).round(2)


### Step back to overall epoch patterns

Below, each point is a subject-level median across 16 trials, not an independent trial observation.


In [ ]:
# Options: any (feature column, axis label) pairs from section_3.core_feature_columns.
# features.OVERVIEW_FEATURES = [
#     ("freezing_fraction", "Freezing fraction"),
#     ("motion_median", "Median motion (a.u.)"),
#     ("speed_median_px_s", "Median speed (px/s)"),
#     ("hr_median_bpm", "Median HR (bpm)"),
# ]
section_3.overview_features = features.OVERVIEW_FEATURES
features.plot_epoch_overview(section_3.extracted_epoch_features, feature_specs=section_3.overview_features)
plt.show()


**Try:** Change the selected trial. Which features strike you as interesting?


## **Section 4: Individual extinction trajectories**

**Goal:** Preserve ordered trials and identify how subjects differ across two extinction days.


### Choose one temporal view

Keep the feature and epoch constant while moving from cohort to individual views.


In [ ]:
section_4 = SimpleNamespace()
# Options: freezing_fraction, longest_freezing_bout_s, latency_to_freezing_s,
# motion_median, motion_iqr, motion_p90, speed_median_px_s, distance_px,
# speed_p90_px_s, hr_median_bpm, hr_p10_bpm, hr_p90_bpm,
# or hr_p90_minus_p10_bpm.
section_4.feature = "freezing_fraction"
# Options: "Pre-Tone", "Early-Tone", "Late-Tone", or "Post-Tone".
section_4.epoch = "Late-Tone"
# Subject selection: "all", one subject ID, or a list of subject IDs.
section_4.subject_selection = "all"
section_4.selected_features = session_data.select_subjects(prepared_epoch_features, section_4.subject_selection)


### Preserve every trial

Look for gradual change, abrupt transitions, persistent levels, reversals, and missing measurements.


In [ ]:
trajectories.plot_trial_heatmap(section_4.selected_features, feature=section_4.feature, epoch=section_4.epoch)
plt.show()


### Follow one subject across both days


In [ ]:
# Focus subject: one subject from subject_selection to inspect longitudinally.
section_4.focus_subject = "subject_01"
if section_4.focus_subject not in section_4.selected_features["subject_id"].unique():
    raise ValueError("focus_subject must be present in section_4.selected_features.")
trajectories.plot_subject_trajectory(section_4.selected_features, section_4.focus_subject, section_4.feature, section_4.epoch)
plt.show()


### Compare subjects on common axes

Small multiples preserve individual paths without an unreadable overlay.


In [ ]:
trajectories.plot_subject_trajectories(section_4.selected_features, 
                                       feature=section_4.feature, 
                                       epoch=section_4.epoch)
plt.show()


### Summarize temporal traits

- Early and late block medians: avg response first and last 4 trials
- Linear trend per trial: overall direction per day
- Median absolute trial change: local instability
- Day-2 early minus day-1 late: between-day shift

In [ ]:
section_4.trajectory_summary = trajectories.summarize_trajectories(section_4.selected_features, 
                                                                   section_4.feature, 
                                                                   section_4.epoch)

section_4.summary_columns = [
    "subject_id",
    "phase",
    "early_block_median",
    "late_block_median",
    "late_minus_early",
    "linear_trend_per_trial",
    "median_absolute_trial_change",
    "day2_early_minus_day1_late",
    "valid_trial_count",
    "early_block_valid_count",
    "late_block_valid_count",
]
section_4.selected_subject_summary = section_4.trajectory_summary.loc[
    section_4.trajectory_summary["subject_id"] == section_4.focus_subject,
    section_4.summary_columns,
]
section_4.selected_subject_summary.round(3)


**Try:** Change the feature, epoch, cohort, or focus subject. Do modalities show the same temporal trends?


## **Section 5: Multivariate patterns: profiles or continuum?**

**Goal:** Combine selected Section 4 temporal traits across modalities, with one row per subject, and ask whether variation appears continuous or separated.


### Build one row per subject

And inspecting missing traits:


In [ ]:
section_5 = SimpleNamespace()
# Subject selection: "all" or a list containing at least two subject IDs.
section_5.subject_selection = "all"
# Options: one or more of "freezing", "longest_bout", "motion", "speed", or "hr".
# Examples: ("freezing", "motion"), ("freezing", "speed"), or ("freezing", "hr").
section_5.signals = ("freezing", "hr")
# Options: "Pre-Tone", "Early-Tone", "Late-Tone", or "Post-Tone".
section_5.epoch = "Late-Tone"

section_5.selected_features = session_data.select_subjects(prepared_epoch_features, section_5.subject_selection)
section_5.pattern_matrix_all = patterns.build_pattern_matrix(section_5.selected_features, section_5.signals, section_5.epoch)
section_5.pattern_availability = pd.DataFrame(
    {
        "missing_traits": section_5.pattern_matrix_all.isna().sum(axis=1),
        "included": ~section_5.pattern_matrix_all.isna().any(axis=1),
    }
)
section_5.pattern_availability


In [ ]:
section_5.pattern_matrix = section_5.pattern_matrix_all.dropna().copy()
section_5.pattern_matrix.round(3)


### Inspect redundancy

Correlated traits can overweight one dimension. Freezing and motion are not fully independent measurements.


In [ ]:
# Options: any positive integer up to the number of unique feature pairs.
section_5.top_correlation_n = 10
section_5.top_correlations = patterns.top_feature_correlations(section_5.pattern_matrix, top_n=section_5.top_correlation_n)
display(section_5.top_correlations.round(3))

patterns.plot_feature_correlation(section_5.pattern_matrix)
plt.show()


### Put traits on a common scale

Z-standardization levels out different scales, preventing any unreal artifacts/similarity.


In [ ]:
section_5.standardized_patterns = patterns.standardize_pattern_matrix(section_5.pattern_matrix)
section_5.standardization_check = pd.DataFrame(
    {
        "mean": section_5.standardized_patterns.mean(),
        "standard_deviation": section_5.standardized_patterns.std(ddof=0),
    }
).round(2)
section_5.standardization_check


### Look for continuous dimensions

PCA:


In [ ]:
section_5.pca_scores, section_5.pca_loadings, section_5.explained_variance = patterns.fit_pattern_pca(section_5.standardized_patterns)
patterns.plot_pattern_pca(section_5.pca_scores, section_5.pca_loadings, section_5.explained_variance)
plt.show()


### Ask whether similarity forms branches

Let's adopt a simple approach to inspect hierarchies in our data:


In [ ]:
patterns.plot_hierarchical_patterns(section_5.standardized_patterns)
plt.show()


### Pool all epochs as repeated extinction states

UMAP provides an exploratory map of locally similar trial-epoch states. We can look for gradients, recurring regions, and possible groupings, while remembering that repeated observations from the same subject are related rather than independent.


In [ ]:
# Options include freezing_fraction, longest_freezing_bout_s, latency_to_freezing_s,
# motion_median, motion_iqr, motion_p90, speed_median_px_s, distance_px,
# speed_p90_px_s, hr_median_bpm, hr_p10_bpm, hr_p90_bpm, or hr_p90_minus_p10_bpm.
# section_5.state_features = ("freezing_fraction", "longest_freezing_bout_s", "motion_median", "speed_median_px_s")
section_5.state_features = ("freezing_fraction", "distance_px", "hr_median_bpm")
# Options: an integer from 2 to one less than the included observation count.
section_5.umap_neighbors = 30
# Options: a number from 0.0 for compact neighbourhoods to 1.0 for greater spread.
section_5.umap_min_dist = 0.20
# Options: any integer; retain 42 when exact workshop reproducibility is preferred.
section_5.umap_random_state = 42

section_5.state_matrix_all, section_5.state_metadata_all = patterns.build_state_matrix(section_5.selected_features, section_5.state_features)
section_5.complete_states = ~section_5.state_matrix_all.isna().any(axis=1)
section_5.state_availability = section_5.state_metadata_all[["subject_id"]].copy()
section_5.state_availability["included"] = section_5.complete_states
section_5.state_availability = section_5.state_availability.groupby("subject_id")["included"].agg(["sum", "count"])
section_5.state_availability.columns = ["included_observations", "total_observations"]
section_5.state_availability["excluded_observations"] = section_5.state_availability["total_observations"] - section_5.state_availability["included_observations"]
display(section_5.state_availability)

section_5.state_matrix = section_5.state_matrix_all.loc[section_5.complete_states].copy()
section_5.state_metadata = section_5.state_metadata_all.loc[section_5.complete_states].copy()
section_5.standardized_states = patterns.standardize_pattern_matrix(section_5.state_matrix)
section_5.state_embedding, section_5.state_diagnostics = patterns.fit_state_umap(section_5.standardized_states, n_neighbors=section_5.umap_neighbors, min_dist=section_5.umap_min_dist, random_state=section_5.umap_random_state)
section_5.state_diagnostics["excluded_observations"] = len(section_5.state_matrix_all) - len(section_5.state_matrix)
display(section_5.state_diagnostics.round(3))
patterns.plot_state_umap(section_5.state_embedding, section_5.state_metadata)
plt.show()


**Decision:** Do visible regions follow epoch, day, trial, or subject? Do not name phenotypes yet.
